# Molecule-Stage PPO On Kaggle

This notebook resolves grouped splits and the best multi-molecule SFT checkpoint from Kaggle artifacts or local outputs, then continues the downstream pipeline that starts from the diverse-beam collection stage and exports a clean PPO artifact bundle.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "train_molecule_wise_ppo"
UPSTREAM_MULTI_SFT_STAGE = "train_multi_molecule_sft"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    copy_stage_artifact_to_local,
    ensure_paths_exist,
    ensure_runtime_dependencies,
    dump_yaml,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    report_runtime,
)


In [ ]:
PPO_ITERATIONS = 20
BATCH_SIZE = 16
MINI_BATCH_SIZE = 4
PPO_EPOCHS_PER_BATCH = 2
SAVE_EVERY_ITERATIONS = 5
LOCAL_SPLIT_DIR = REPO_DIR / "kaggle" / "generated_data" / "post_training_splits"
LOCAL_BEST_CHECKPOINT = REPO_DIR / "outputs" / "kaggle" / "multi_molecule_sft" / "checkpoints" / "best"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "molecule_wise_ppo"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / "molecule_wise_ppo.kaggle.yaml"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "output_dir": str(OUTPUT_DIR),
    "local_split_dir": str(LOCAL_SPLIT_DIR),
    "local_best_checkpoint": str(LOCAL_BEST_CHECKPOINT),
}))


In [ ]:
required_split_paths = {
    "train": LOCAL_SPLIT_DIR / "train_multimol.jsonl",
    "validation": LOCAL_SPLIT_DIR / "validation_multimol.jsonl",
    "test": LOCAL_SPLIT_DIR / "test_multimol.jsonl",
}
if not all(path.exists() for path in required_split_paths.values()):
    copied_splits = copy_stage_artifact_to_local(
        stage_name=UPSTREAM_MULTI_SFT_STAGE,
        artifact_relpath="grouped_splits",
        local_path=LOCAL_SPLIT_DIR,
    )
else:
    copied_splits = None

if not LOCAL_BEST_CHECKPOINT.exists():
    copied_checkpoint = copy_stage_artifact_to_local(
        stage_name=UPSTREAM_MULTI_SFT_STAGE,
        artifact_relpath="checkpoints/best",
        local_path=LOCAL_BEST_CHECKPOINT,
    )
else:
    copied_checkpoint = None

if not all(path.exists() for path in required_split_paths.values()):
    raise FileNotFoundError(
        f"Grouped split files are missing under {LOCAL_SPLIT_DIR}. Attach the multi-molecule SFT artifact dataset or run 02_train_multi_molecule_sft.ipynb first."
    )
if not LOCAL_BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best multi-molecule SFT checkpoint is missing at {LOCAL_BEST_CHECKPOINT}. Attach the multi-molecule SFT artifact dataset or run 02_train_multi_molecule_sft.ipynb first."
    )

print(json_dumps({
    "copied_splits": None if copied_splits is None else str(copied_splits),
    "copied_checkpoint": None if copied_checkpoint is None else str(copied_checkpoint),
    "split_paths": {name: str(path) for name, path in required_split_paths.items()},
    "checkpoint": str(LOCAL_BEST_CHECKPOINT),
}))

config = load_yaml(REPO_DIR / "configs" / "molecule_wise_ppo.yaml")
config["model"]["checkpoint"] = str(LOCAL_BEST_CHECKPOINT)
config["data"]["train_file"] = str(required_split_paths["train"])
config["data"]["validation_file"] = str(required_split_paths["validation"])
config["data"]["test_file"] = str(required_split_paths["test"])
config["training"]["output_dir"] = str(OUTPUT_DIR)
config["training"]["save_every_iterations"] = int(SAVE_EVERY_ITERATIONS)
config["ppo"]["ppo_iterations"] = int(PPO_ITERATIONS)
config["ppo"]["batch_size"] = int(BATCH_SIZE)
config["ppo"]["mini_batch_size"] = int(MINI_BATCH_SIZE)
config["ppo"]["ppo_epochs_per_batch"] = int(PPO_EPOCHS_PER_BATCH)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/train_molecule_wise_ppo.py --config "{TEMP_CONFIG_PATH}"


In [ ]:
required_outputs = ensure_paths_exist({
    "output_dir": OUTPUT_DIR,
    "checkpoints": OUTPUT_DIR / "checkpoints",
    "run_summary": OUTPUT_DIR / "run_summary.json",
    "history": OUTPUT_DIR / "history.json",
    "resolved_config": OUTPUT_DIR / "resolved_config.yaml",
    "reward_config": OUTPUT_DIR / "reward_config.json",
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "checkpoints": OUTPUT_DIR / "checkpoints",
        "run_summary.json": OUTPUT_DIR / "run_summary.json",
        "history.json": OUTPUT_DIR / "history.json",
        "resolved_config.yaml": OUTPUT_DIR / "resolved_config.yaml",
        "reward_config.json": OUTPUT_DIR / "reward_config.json",
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "summary": read_json(OUTPUT_DIR / "run_summary.json"),
    },
)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
}))
